# In-Class Assignment — Python for Feature Extraction
**Time:** 20 minutes  |  **Points:** 20


....


Dataset file: `product_reviews.txt`

## Load the dataset
Download it from **Canvas**, then run the upload cell below to select the file from your computer.


In [4]:
# Upload the dataset file from your computer (Google Colab)
from google.colab import files

uploaded = files.upload()   # choose product_reviews.txt
filename = next(iter(uploaded))
print("Uploaded file name:", filename)


Saving product_reviews.txt to product_reviews (3).txt
Uploaded file name: product_reviews (3).txt


## Q1 (2 point) — Load & Read & inspect

## Note about the header
**Important:** The dataset file includes a **header row** (column names).  
Make sure the header is used as the column names — it **should NOT appear as a data row** in your DataFrame.

Tip: Use the filename variable printed above when reading the file.
After Reading, check that your columns are `id` and `text`.
Print: **(a)** `df.shape`, **(b)** `df.head(3\)`.  


In [5]:
import pandas as pd

df = pd.read_csv(filename, sep="\t")

print(df.shape)
print(df.head(3))



(10, 2)
   id                                               text
0   1  Love this blender! Smoothies are super creamy ...
1   2  Terrible quality... stopped working after 2 da...
2   3      Good value for the price. Shipping was quick.


## Q2 (4 points) — Basic handcrafted features  
Create these columns and then display the DataFrame:
- `word_count` = number of words  
- `char_count` = number of characters  
- `avg_word_len` = average word length (ignore punctuation)  
- `excl_count` = number of `!` characters  

Print: `id, word_count, char_count, avg_word_len, excl_count`.


In [6]:
import re
import numpy as np

def clean_words(text):
    return re.findall(r"\b\w+\b", str(text).lower())

df["word_count"] = df["text"].apply(lambda x: len(str(x).split()))
df["char_count"] = df["text"].apply(lambda x: len(str(x)))
df["avg_word_len"] = df["text"].apply(
    lambda x: round(np.mean([len(w) for w in clean_words(x)]), 2) if clean_words(x) else 0
)
df["excl_count"] = df["text"].apply(lambda x: str(x).count("!"))

print(df[["id", "word_count", "char_count", "avg_word_len", "excl_count"]])



   id  word_count  char_count  avg_word_len  excl_count
0   1           9          55          5.00           1
1   2           7          51          5.57           3
2   3           8          45          4.50           0
3   4          10          56          4.50           0
4   5           8          53          5.50           1
5   6          10          51          4.00           0
6   7           9          50          4.44           0
7   8           6          40          5.50           0
8   9           7          52          6.29           0
9  10           7          48          4.75           0


## Q3 (6 points) — Bag-of-Words (CountVectorizer)  
Use `CountVectorizer(stop_words="english")` on `df["text"]`. Print:
1) vocabulary size (number of features)  
2) top 10 words by **total count** across all documents (word + count)


In [7]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

vectorizer = CountVectorizer(stop_words="english")
X = vectorizer.fit_transform(df["text"])

words = np.array(vectorizer.get_feature_names_out())
counts = np.array(X.sum(axis=0)).flatten()

top_10 = sorted(zip(words, counts), key=lambda x: (-x[1], x[0]))[:10]

print("Vocabulary size:", len(words))
print("Top 10 words by total count:")
for word, count in top_10:
    print(word, count)



Vocabulary size: 50
Top 10 words by total count:
amazing 1
app 1
battery 1
blender 1
box 1
buy 1
charged 1
clear 1
crashing 1
creamy 1


## Q4 (4 points) — Bigram features  
Use `CountVectorizer(stop_words="english", ngram_range=(2,2))`.  
Print the top 5 bigrams by total count (bigram + count).


In [8]:
bigram_vectorizer = CountVectorizer(stop_words="english", ngram_range=(2, 2))
X_bigram = bigram_vectorizer.fit_transform(df["text"])

bigrams = np.array(bigram_vectorizer.get_feature_names_out())
bigram_counts = np.array(X_bigram.sum(axis=0)).flatten()

top_5_bigrams = sorted(zip(bigrams, bigram_counts), key=lambda x: (-x[1], x[0]))[:5]

print("Top 5 bigrams by total count:")
for bigram, count in top_5_bigrams:
    print(bigram, count)


Top 5 bigrams by total count:
amazing screen 1
app keeps 1
battery life 1
blender smoothies 1
box damaged 1


## Q5 (4 points) — TF-IDF features  
Use `TfidfVectorizer(stop_words="english", ngram_range=(1,2))`.  
Compute the **average TF-IDF** score of each term across documents and print the top 5 terms (term + avg score).


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
X_tfidf = tfidf_vectorizer.fit_transform(df["text"])

terms = np.array(tfidf_vectorizer.get_feature_names_out())
avg_tfidf = np.array(X_tfidf.mean(axis=0)).flatten()

top_5_tfidf = sorted(zip(terms, avg_tfidf), key=lambda x: (-x[1], x[0]))[:5]

print("Top 5 terms by average TF-IDF:")
for term, score in top_5_tfidf:
    print(term, round(score, 4))



Top 5 terms by average TF-IDF:
clear 0.0378
easy 0.0378
easy instructions 0.0378
instructions 0.0378
instructions clear 0.0378


## Grading Checklist
- Q1: correct prints  
- Q2: correct feature columns + requested display  
- Q3: correct vocabulary size + correct top 10 words by total count  
- Q4: correct top 5 bigrams by total count  
- Q5: correct top 5 TF-IDF terms by average score
